In [2]:
# 计算文本的表征，存成文件
from transformers import AutoTokenizer,AutoModel
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset

model_name = "princeton-nlp/unsup-simcse-bert-base-uncased"
dataset_name = "wiki1m_for_simcse.txt"
dataset = load_dataset("LyuShawn/Dataset-LyuCSE", data_files=dataset_name)
dataset = dataset['train']

output_file = f"data/emb_{model_name.split('/')[-1]}_{dataset_name.split('.')[0]}.npy"

# 采样1000个样本
dataset = dataset.shuffle(seed=42).select(range(10000))
max_seq_length = 32
bs = 1024    # 以bs为单位进行推理
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

# 用于存储所有文本的表征
all_embeddings = []

def prepare(examples):

    return tokenizer(examples["text"], padding=False, truncation=True, max_length=max_seq_length)

dataset = dataset.map(prepare, batched=True)


for batch in tqdm(dataset.batch(bs)):
    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]

    # 对齐
    max_len = max([len(ids) for ids in input_ids])
    input_ids = [ids + [tokenizer.pad_token_id] * (max_len - len(ids)) for ids in input_ids]
    attention_mask = [mask + [0] * (max_len - len(mask)) for mask in attention_mask]

    input_ids = torch.tensor(input_ids, dtype=torch.long).to(device)
    attention_mask = torch.tensor(attention_mask, dtype=torch.long).to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)

    last_hidden_state = outputs.last_hidden_state
    pooler_output =last_hidden_state[:,0,:]
    all_embeddings.append(pooler_output.cpu().numpy())

# 将所有的文本表征和成一个array
all_embeddings = np.concatenate(all_embeddings, axis=0)
np.save(output_file, all_embeddings)
print(f"Saved to {output_file}")

100%|██████████| 10/10 [00:05<00:00,  1.78it/s]

Saved to data/emb_unsup-simcse-bert-base-uncased_wiki1m_for_simcse.npy


In [ ]:
# 统计句子长度分布

output_file = 
input_file = "../data/wiki1m_for_simcse.txt"
with open(input_file, 'r', encoding='utf-8') as f:
    sent_list = f.read().splitlines()